# ArtMarket AI: Fine-Tuning BLIP for Artistic Image Captioning 🖌️

Welcome! This notebook will help you train a custom version of the BLIP model specifically for your art style. 

### Why fine-tune?
General AI models like BLIP are trained on everything (photos, cats, cars). By fine-tuning on *your* art, the AI will learn a specific vocabulary: the way you use light, your specific brush strokes, and the emotions in your work.

---

## 1. Setup Environment

In [ ]:
!pip install -q transformers[torch] datasets torch pillow tqdm huggingface_hub

## 2. Prepare Data

To train your model, you need two things:
1. A folder called `images` containing your JPEGs/PNGs.
2. A file called `metadata.jsonl` where each line looks like this:
`{"file_name": "drawing_01.jpg", "text": "A charcoal sketch of a lonely figure in a rainy city."}`

**Pro Tip:** Use the code below to upload your data.

In [ ]:
from google.colab import files
import os

print("Upload your metadata.jsonl file first:")
uploaded = files.upload()

os.makedirs('images', exist_ok=True)
print("Now upload multiple images to the 'images/' folder using the sidebar (folder icon on the left).")

## 3. Training Logic

This is the core training script. We'll load the Salesforce BLIP model and fine-tune it on your artwork.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json
from transformers import BlipProcessor, BlipForConditionalGeneration, AdamW
from tqdm import tqdm

class ArtDataset(Dataset):
    def __init__(self, metadata_path, image_dir, processor):
        self.data = []
        with open(metadata_path, 'r') as f:
            for line in f:
                self.data.append(json.loads(line))
        self.image_dir = image_dir
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(os.path.join(self.image_dir, item['file_name'])).convert("RGB")
        text = item['text']
        
        encoding = self.processor(images=image, text=text, padding="max_length", return_tensors="pt")
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        return encoding

def train_model(epochs=5, batch_size=2):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

    dataset = ArtDataset("metadata.jsonl", "images", processor)
    train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=5e-5)

    model.train()
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        for batch in tqdm(train_dataloader):
            optimizer.zero_grad()
            
            input_ids = batch.pop("input_ids").to(device)
            pixel_values = batch.pop("pixel_values").to(device)

            outputs = model(input_ids=input_ids, pixel_values=pixel_values, labels=input_ids)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
    
    model.save_pretrained("art_model_final")
    processor.save_pretrained("art_model_final")
    print("Training complete! Saved to ./art_model_final")
    return model, processor

print("Training functions defined. Run this after uploading data to start training.")

## 4. Test Your Model

Let's see the results!

In [ ]:
def generate_caption(image_path, model, processor):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    
    out = model.generate(**inputs)
    return processor.decode(out[0], skip_special_tokens=True)

print("Test function loaded.")

## 5. Export to Hugging Face

To use this in your ArtMarket app, upload it to the Hub!

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Enter your username and desired model name below
# model.push_to_hub("your-user/artmarket-blip-v1")